In [1]:
# Throw a ball at 100 different velocities.

import jax
import mujoco
from mujoco import mjx
from PIL import Image

XML = r"""
<mujoco>
  <worldbody>
    <body>
      <freejoint/>
      <geom size=".15" mass="1" type="sphere"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(XML)
mjx_model = mjx.put_model(model)


@jax.vmap
def batched_step(vel):
    mjx_data = mjx.make_data(mjx_model)
    qvel = mjx_data.qvel.at[0].set(vel)
    mjx_data = mjx_data.replace(qvel=qvel)
    pos = mjx.step(mjx_model, mjx_data).qpos[0]
    return pos


vel = jax.numpy.arange(0.0, 1.0, 0.01)
pos = jax.jit(batched_step)(vel)

In [2]:
mjx_model.opt.timestep

Array(0.002, dtype=float32)

In [3]:
0.01 % 0.002

0.0

In [4]:
data = mjx.make_data(mjx_model)

print("number of position coordinates:", mjx_model.nq)
print("number of velocity coordinates:", mjx_model.nv)

print("joint positions:", data.qpos)
print("joint velocities:", data.qvel)
print("joint accelerations:", data.qacc)

print("body positions:", data.xpos)
print("body rotations:", data.xquat)
print("number of bodies:", mjx_model.nbody)

number of position coordinates: 7
number of velocity coordinates: 6
joint positions: [0. 0. 0. 1. 0. 0. 0.]
joint velocities: [0. 0. 0. 0. 0. 0.]
joint accelerations: [0. 0. 0. 0. 0. 0.]
body positions: [[0. 0. 0.]
 [0. 0. 0.]]
body rotations: [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
number of bodies: 2


In [5]:
import mujoco
import numpy as np

# import media # If you're using Google Colab or similar for image display

chaotic_pendulum = r"""
<mujoco>
  <option timestep=".01" integrator="RK4"/>

  <default>
    <joint type="hinge" axis="0 -1 0" limited="false" range="-150 150"/>
    <geom type="capsule" size=".02" solref="-1 -1"/>
  </default>

  <worldbody>
    <light pos="0 -.4 1"/>
    <camera name="fixed" pos="0 -1.5 0" xyaxes="1 0 0 0 0 1"/>

    <body name="red" pos="0 0 .2">
      <joint name="root" />
      <geom fromto="0 0 0 0 0 -.25" rgba="1 0 0 1"/>
      <body name="green" pos="0 0 -.25">
        <joint/>
        <geom fromto="0 0 0 0 0 -.25" rgba="0 1 0 1"/>
        <body name="blue" pos="0 0 -.25">
          <joint/>
          <geom fromto="0 0 0 0 0 -.25" rgba="0 0 1 1"/>
          <body name="purple" pos="0 0 -.25">
            <joint/>
            <geom fromto="0 0 0 0 0 -.25" rgba="0 1 1 1"/>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <motor name="joint_motor" joint="root"/>
  </actuator>
</mujoco>
"""

# Load the MuJoCo model from the XML string
model = mujoco.MjModel.from_xml_string(chaotic_pendulum)
# Create initial CPU-based mjData
data = mujoco.MjData(model)

# Set up rendering options
height = 480
width = 640
scene_option = mujoco.MjvOption()
# Enable visualization of joints in the renderer
scene_option.flags[mujoco.mjtVisFlag.mjVIS_JOINT] = True

# Get the actuator ID for the "joint_motor"
actuator_id = model.actuator("joint_motor").id

# --- Simulation with rendering to images ---
# Create a renderer object for visualizing the simulation
renderer = mujoco.Renderer(model, height, width)

duration = 5  # seconds: Total simulation time
framerate = 60  # frames per second: Desired GIF framerate
# Calculate total simulation steps based on duration and model timestep
num_simulation_steps = int(duration / model.opt.timestep)
# Calculate how many simulation steps correspond to one rendered frame
render_interval = int(1 / (framerate * model.opt.timestep))

frames = []  # List to store rendered image arrays

print("Starting simulation and rendering with CPU MuJoCo...")
for i in range(num_simulation_steps):
    # Define a sinusoidal torque command
    torque_amplitude = 5.0
    torque_frequency = 2.0
    torque_value = torque_amplitude * np.sin(
        torque_frequency * i * model.opt.timestep
    )

    # Apply the torque to the actuator in mjData (CPU-based)
    data.ctrl[actuator_id] = torque_value

    # Advance the CPU MuJoCo simulation by one step
    mujoco.mj_step(model, data)

    # Render a frame at the specified interval
    if i % render_interval == 0:
        # Use the CPU-based 'data' directly for rendering
        renderer.update_scene(data, camera="fixed", scene_option=scene_option)
        pixels = renderer.render()
        frames.append(pixels)

print(f"Simulation finished. Captured {len(frames)} frames.")

# Save the captured frames as a GIF
images = [Image.fromarray(f) for f in frames]
images[0].save(
    "pendulum_animation.gif",
    save_all=True,
    append_images=images[1:],
    optimize=True,
    duration=model.opt.timestep
    * 1000
    * render_interval,  # Duration per frame in ms
    loop=0,  # Loop indefinitely
)
print("GIF saved as 'pendulum_animation.gif'")

# Clean up the renderer resources
renderer.close()


Starting simulation and rendering with CPU MuJoCo...
Simulation finished. Captured 500 frames.
GIF saved as 'pendulum_animation.gif'


In [6]:
print("number of position coordinates:", model.nq)
print("number of velocity coordinates:", model.nv)
print("number of actuators:", model.nu)

data = mujoco.MjData(model)
mujoco.mj_step(model, data)

print("simulation time:", data.time)
print("joint positions:", data.qpos)
print("joint velocities:", data.qvel)
print("joint accelerations:", data.qacc)

print("body positions:", data.xpos)

number of position coordinates: 4
number of velocity coordinates: 4
number of actuators: 1
simulation time: 0.01
joint positions: [0. 0. 0. 0.]
joint velocities: [0. 0. 0. 0.]
joint accelerations: [0. 0. 0. 0.]
body positions: [[ 0.    0.    0.  ]
 [ 0.    0.    0.2 ]
 [ 0.    0.   -0.05]
 [ 0.    0.   -0.3 ]
 [ 0.    0.   -0.55]]


In [7]:
data.ctrl

array([0.])